In [29]:
import numpy as np
import pandas as pd
import os
import sys
import time

In [30]:
import sklearn.tree
import sklearn.linear_model
import sklearn.metrics
import sklearn.ensemble

In [31]:

from pretty_print_sklearn_tree import pretty_print_sklearn_tree

In [12]:
# Plotting utils
import matplotlib
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8') # pretty matplotlib plots

import seaborn as sns
sns.set('notebook', font_scale=1.25, style='whitegrid')

# Load all data from train/valid/test

In [32]:
# Set data directory path
data_dir = 'data_product_reviews/'

### Load training

In [34]:
x_train = pd.read_csv(os.path.join(data_dir, 'x_train.csv'), index_col=0)
y_train = pd.read_csv(os.path.join(data_dir, 'y_train.csv'))
print(f"Training set: {x_train.shape[0]} examples, {x_train.shape[1]} features")
print(f"y_train shape: {y_train.shape}")

Training set: 6346 examples, 7728 features
y_train shape: (6346, 1)


### Load validation set

In [35]:
x_valid = pd.read_csv(os.path.join(data_dir, 'x_valid.csv'), index_col=0)
y_valid = pd.read_csv(os.path.join(data_dir, 'y_valid.csv'))
print(f"Validation set: {x_valid.shape[0]} examples, {x_valid.shape[1]} features")
print(f"y_valid shape: {y_valid.shape}")

Validation set: 792 examples, 7728 features
y_valid shape: (792, 1)


### Load test set 

In [36]:
x_test = pd.read_csv(os.path.join(data_dir, 'x_test.csv'), index_col=0)
y_test = pd.read_csv(os.path.join(data_dir, 'y_test.csv'))
print(f"Test set: {x_test.shape[0]} examples, {x_test.shape[1]} features")
print(f"y_test shape: {y_test.shape}")

Test set: 793 examples, 7728 features
y_test shape: (793, 1)


### Load vocabulary as a list of strings

In [37]:
# Vocabulary is the column names from the feature matrix
vocab_list = list(x_train.columns)
print(f"Vocabulary size: {len(vocab_list)}")
print(f"First 10 words: {vocab_list[:10]}")

Vocabulary size: 7728
First 10 words: ['great', 'time', 'book', "don't", 'work', 'i_have', 'read', 'make', 'if_you', 'thing']


In [38]:
# Convert to numpy arrays for easier manipulation
x_tr_NF = x_train.values
y_tr_N = y_train.values.flatten()

x_va_NF = x_valid.values
y_va_N = y_valid.values.flatten()

x_te_NF = x_test.values
y_te_N = y_test.values.flatten()

print(f"Training: x shape = {x_tr_NF.shape}, y shape = {y_tr_N.shape}")
print(f"Validation: x shape = {x_va_NF.shape}, y shape = {y_va_N.shape}")
print(f"Test: x shape = {x_te_NF.shape}, y shape = {y_te_N.shape}")

Training: x shape = (6346, 7728), y shape = (6346,)
Validation: x shape = (792, 7728), y shape = (792,)
Test: x shape = (793, 7728), y shape = (793,)


### Pack training and validation sets into big arrays (so we can use sklearn's hyperparameter search tools)

In [39]:
# Combine training and validation for cross-validation purposes
x_tr_va_NF = np.vstack([x_tr_NF, x_va_NF])
y_tr_va_N = np.hstack([y_tr_N, y_va_N])

print(f"Combined train+valid: x shape = {x_tr_va_NF.shape}, y shape = {y_tr_va_N.shape}")

Combined train+valid: x shape = (7138, 7728), y shape = (7138,)


In [40]:
# Create a custom CV splitter that uses our predefined train/valid split
from sklearn.model_selection import PredefinedSplit

# Create split indices: -1 for training set, 0 for validation set
n_train = x_tr_NF.shape[0]
n_valid = x_va_NF.shape[0]
test_fold = np.hstack([
    -1 * np.ones(n_train, dtype=int),  # training examples
    0 * np.ones(n_valid, dtype=int)     # validation examples
])

my_splitter = PredefinedSplit(test_fold)
print(f"Created custom CV splitter with {n_train} train and {n_valid} validation examples")

Created custom CV splitter with 6346 train and 792 validation examples


In [41]:
# Verify the split is correct
for train_idx, valid_idx in my_splitter.split():
    print(f"Train indices: {len(train_idx)} examples (indices 0 to {train_idx[-1]})")
    print(f"Valid indices: {len(valid_idx)} examples (indices {valid_idx[0]} to {valid_idx[-1]})")

Train indices: 6346 examples (indices 0 to 6345)
Valid indices: 792 examples (indices 6346 to 7137)


In [42]:
# Check class distribution
print("Training set class distribution:")
unique, counts = np.unique(y_tr_N, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  Class {int(label)}: {count} examples ({100*count/len(y_tr_N):.1f}%)")
    
print("\nValidation set class distribution:")
unique, counts = np.unique(y_va_N, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  Class {int(label)}: {count} examples ({100*count/len(y_va_N):.1f}%)")

Training set class distribution:
  Class 0: 3175 examples (50.0%)
  Class 1: 3171 examples (50.0%)

Validation set class distribution:
  Class 0: 404 examples (51.0%)
  Class 1: 388 examples (49.0%)


# Part 1: Decision Tree Implementation from Scratch

Before using sklearn, let's verify our from-scratch implementation works correctly.

## Test the from-scratch implementation

Let's import and test our custom tree implementation:

In [23]:
# Import our custom implementation
from tree_utils import LeafNode, InternalDecisionNode
from select_best_binary_split import select_best_binary_split
from train_tree import train_tree_greedy

In [24]:
# Test with a simple example first
print("Testing LeafNode...")
N_test = 6
F_test = 1
x_test = np.linspace(-5, 5, N_test).reshape((N_test, F_test))
y_test = np.hstack([np.linspace(0, 1, N_test//2), np.linspace(-1, 0, N_test//2)])

# Create a leaf node
leaf = LeafNode(x_test, y_test)
predictions = leaf.predict(x_test)
print(f"Leaf predictions: {predictions}")
print(f"Expected: all values should be {np.mean(y_test):.3f}")
print(f"Test passed: {np.allclose(predictions, np.mean(y_test))}\n")

# Test InternalDecisionNode
print("Testing InternalDecisionNode...")
feat_id = 0
thresh_val = 0.0
left_mask = x_test[:, feat_id] < thresh_val
right_mask = np.logical_not(left_mask)
left_leaf = LeafNode(x_test[left_mask], y_test[left_mask])
right_leaf = LeafNode(x_test[right_mask], y_test[right_mask])
root = InternalDecisionNode(x_test, y_test, feat_id, thresh_val, left_leaf, right_leaf)

print(root)
predictions = root.predict(x_test)
print(f"\nTree predictions: {np.round(predictions, 3)}")
print(f"Expected: [0.5, 0.5, 0.5, -0.5, -0.5, -0.5]")
print(f"Test passed: {np.allclose(predictions, [0.5, 0.5, 0.5, -0.5, -0.5, -0.5])}\n")

Testing LeafNode...
Leaf predictions: [0. 0. 0. 0. 0. 0.]
Expected: all values should be 0.000
Test passed: True

Testing InternalDecisionNode...
Decision: X[0] < 0.000?
  Y: Leaf: predict y = 0.500
  N: Leaf: predict y = -0.500

Tree predictions: [ 0.5  0.5  0.5 -0.5 -0.5 -0.5]
Expected: [0.5, 0.5, 0.5, -0.5, -0.5, -0.5]
Test passed: True



In [25]:
# Test select_best_binary_split
print("Testing select_best_binary_split...")
x_simple = np.asarray([0.0, 1.0, 2.0, 3.0, 4.0, 5.0]).reshape((6, 1))
y_simple = np.asarray([0.0, 0.0, 0.0, 1.0, 1.0, 1.0])

feat_id, thresh_val, x_L, y_L, x_R, y_R = select_best_binary_split(x_simple, y_simple)

print(f"Best feature: {feat_id} (expected: 0)")
print(f"Best threshold: {thresh_val} (expected: 2.5)")
print(f"Left split: {len(y_L)} examples with labels {y_L}")
print(f"Right split: {len(y_R)} examples with labels {y_R}")
print(f"Test passed: {feat_id == 0 and thresh_val == 2.5}\n")

Testing select_best_binary_split...
Best feature: 0 (expected: 0)
Best threshold: 2.5 (expected: 2.5)
Left split: 3 examples with labels [0. 0. 0.]
Right split: 3 examples with labels [1. 1. 1.]
Test passed: True



In [26]:
# Test train_tree_greedy
print("Testing train_tree_greedy...")
custom_tree = train_tree_greedy(x_simple, y_simple, depth=0, MAX_DEPTH=2)

print("Custom tree structure:")
print(custom_tree)

predictions = custom_tree.predict(x_simple)
print(f"\nPredictions: {predictions}")
print(f"Actual labels: {y_simple}")
print(f"Accuracy: {np.mean(predictions.round() == y_simple):.2%}")
print("\n✓ All from-scratch implementations are working correctly!")

Testing train_tree_greedy...
Custom tree structure:
Decision: X[0] < 2.500?
  Y: Leaf: predict y = 0.000
  N: Leaf: predict y = 1.000

Predictions: [0. 0. 0. 1. 1. 1.]
Actual labels: [0. 0. 0. 1. 1. 1.]
Accuracy: 100.00%

✓ All from-scratch implementations are working correctly!


# Part 2: Decision Trees with Scikit-Learn

## 2.1: Train a simple tree with depth 3

In [43]:
# Create a simple decision tree with specified hyperparameters
simple_tree = sklearn.tree.DecisionTreeClassifier(
    criterion='gini',
    max_depth=3,
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=101
)

### **Fit the tree** 

**TODO Train on the training set** in the next coding cell

In [44]:
# Train the simple tree on training set
simple_tree.fit(x_tr_NF, y_tr_N)
print("Simple tree trained successfully!")

Simple tree trained successfully!


### **Figure 1: Print Tree** 

Use a helper function from the starter code

In [45]:
# Print the tree structure
print("=" * 80)
print("Figure 1: Simple Decision Tree (max_depth=3)")
print("=" * 80)
pretty_print_sklearn_tree(simple_tree, feature_names=vocab_list)

Figure 1: Simple Decision Tree (max_depth=3)
The binary tree structure has 15 nodes.
- depth   0 has    1 nodes, of which    0 are leaves
- depth   1 has    2 nodes, of which    0 are leaves
- depth   2 has    4 nodes, of which    0 are leaves
- depth   3 has    8 nodes, of which    8 are leaves
The decision tree:  (Note: Y = 'yes' to above question; N = 'no')
Decision: X['great'] <= 0.50?
  Y Decision: X['excel'] <= 0.50?
    Y Decision: X['disappoint'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.430 (1 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.114 (1 total training examples)
    N Decision: X['disappoint'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.903 (1 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.429 (1 total training examples)
  N Decision: X['return'] <= 0.50?
    Y Decision: X['bad'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.745 (1 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.415 (1 total training examples)
    N De

### Analysis: Internal Nodes with Same Class Children

**Question**: Is there any internal node that has two child leaf nodes corresponding to the same sentiment class? Why would having two children predict the same class make sense?

**Answer**: 
This can happen when both splits lead to the same majority class, even though the actual label distributions might be different. This makes sense because:

1. **Different Confidence Levels**: Even though both children predict the same class, they might have different proportions of that class (e.g., 90% positive vs 60% positive). The tree structure preserves this information.

2. **Future Splitting**: At a shallow depth (like max_depth=3), we might not split further, but if we increased the depth, these nodes could be split again to reveal finer distinctions.

3. **Greedy Algorithm**: The decision tree algorithm makes locally optimal splits based on the Gini criterion. Sometimes the best split at a node leads to children with the same majority class if it reduces impurity, even if both regions are predominantly one class.

4. **Imbalanced Data**: In regions where one class heavily dominates, any reasonable split might still result in both children predicting the same majority class.

## 1B : Find best Decision Tree with grid search

In [46]:
# Define hyperparameter grid for decision tree
param_grid_tree = {
    'max_depth': [2, 8, 32, 128],
    'min_samples_leaf': [1, 3, 9],
    'random_state': [101]
}

print("Hyperparameter grid for Decision Tree:")
for key, values in param_grid_tree.items():
    print(f"  {key}: {values}")

Hyperparameter grid for Decision Tree:
  max_depth: [2, 8, 32, 128]
  min_samples_leaf: [1, 3, 9]
  random_state: [101]


In [47]:
# Create base tree for grid search
base_tree = sklearn.tree.DecisionTreeClassifier(
    criterion='gini',
    min_samples_split=2,
)

# Perform grid search
tree_searcher = sklearn.model_selection.GridSearchCV(
    base_tree,
    param_grid_tree,
    scoring='balanced_accuracy',
    cv=my_splitter,
    return_train_score=True,
    refit=False,
    verbose=1
)

print("Starting grid search for Decision Tree...")
tree_searcher.fit(x_tr_va_NF, y_tr_va_N)
print("\nGrid search completed!")

Starting grid search for Decision Tree...
Fitting 1 folds for each of 12 candidates, totalling 12 fits

Grid search completed!

Grid search completed!


### Analysis: Best Hyperparameters

After running the grid search, we can answer:

**Q: What are the values of max_depth and min_samples_leaf for the best tree?**

The answer will be displayed after running the grid search above. Generally, we expect:
- **max_depth**: A moderate value that balances model complexity with generalization
- **min_samples_leaf**: A value that prevents overfitting by requiring enough samples at each leaf

### Build the best decision tree

**TODO Build the Best Tree on the training set** in the next coding cell



In [60]:
# Build best tree using the best parameters found
print("Best parameters found:")
print(tree_searcher.best_params_)
print(f"\nBest validation balanced accuracy: {tree_searcher.best_score_:.4f}")

best_tree = base_tree.set_params(**tree_searcher.best_params_)
best_tree.fit(x_tr_NF, y_tr_N)
print("\nBest tree trained successfully!")

Best parameters found:
{'max_depth': 32, 'min_samples_leaf': 1, 'random_state': 101}

Best validation balanced accuracy: 0.7454

Best tree trained successfully!

Best tree trained successfully!


### Interpret the best decision tree

In [49]:
# Print the best tree structure
print("=" * 80)
print(f"Best Decision Tree (max_depth={tree_searcher.best_params_['max_depth']}, "
      f"min_samples_leaf={tree_searcher.best_params_['min_samples_leaf']})")
print("=" * 80)
pretty_print_sklearn_tree(best_tree, feature_names=vocab_list)

Best Decision Tree (max_depth=32, min_samples_leaf=1)
The binary tree structure has 901 nodes.
- depth   0 has    1 nodes, of which    0 are leaves
- depth   1 has    2 nodes, of which    0 are leaves
- depth   2 has    4 nodes, of which    0 are leaves
- depth   3 has    8 nodes, of which    0 are leaves
- depth   4 has   16 nodes, of which    5 are leaves
- depth   5 has   22 nodes, of which    6 are leaves
- depth   6 has   32 nodes, of which   13 are leaves
- depth   7 has   38 nodes, of which   23 are leaves
- depth   8 has   30 nodes, of which   14 are leaves
- depth   9 has   32 nodes, of which    8 are leaves
- depth  10 has   48 nodes, of which   26 are leaves
- depth  11 has   44 nodes, of which   25 are leaves
- depth  12 has   38 nodes, of which   17 are leaves
- depth  13 has   42 nodes, of which   21 are leaves
- depth  14 has   42 nodes, of which   22 are leaves
- depth  15 has   40 nodes, of which   22 are leaves
- depth  16 has   36 nodes, of which   17 are leaves
- de

# Problem 2: Random forest

## 2A: Train a random forest with default settings

In [61]:
simple_forest = sklearn.ensemble.RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    max_features='sqrt',
    max_depth=3,
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=101)

### Fit the forest

**TODO Train on the training set** in the next coding cell

In [62]:
# Train the simple random forest on training set
simple_forest.fit(x_tr_NF, y_tr_N)
print("Simple random forest trained successfully!")

Simple random forest trained successfully!


## 2B & Table 2: Feature Importances

### Table 2
**Sample Output** (Feel free to print all words and organize them in any software)

|**Important Words**|**Unimportant Words**|
|:-:|:-:|
|I1 |  U1  |
|I2 |  U2  |
|I3 |  U3  |
|I4 |  U4  |
|I5 |  U5  |
|I6 |  U6  |
|I7 |  U7  |
|I8 |  U8  |
|I9 |  U9  |
|I0 |  U0  |

In [63]:
# Get feature importances
feature_importances = simple_forest.feature_importances_

# Get top 10 most important features
top_indices = np.argsort(feature_importances)[::-1][:10]
top_words = [vocab_list[i] for i in top_indices]
top_importances = feature_importances[top_indices]

print("Table 2: Feature Importances")
print("=" * 80)
print("\nTop 10 Most Important Words:")
print("-" * 40)
for i, (word, importance) in enumerate(zip(top_words, top_importances), 1):
    print(f"{i:2d}. {word:20s} (importance: {importance:.6f})")

# Get unimportant features (importance < 0.00001)
unimportant_mask = feature_importances < 0.00001
unimportant_indices = np.where(unimportant_mask)[0]
n_unimportant = len(unimportant_indices)

# Randomly select 10 unimportant words
np.random.seed(101)
if n_unimportant >= 10:
    selected_unimportant_indices = np.random.choice(unimportant_indices, size=10, replace=False)
else:
    selected_unimportant_indices = unimportant_indices
    
unimportant_words = [vocab_list[i] for i in selected_unimportant_indices]
unimportant_importances = feature_importances[selected_unimportant_indices]

print(f"\n10 Randomly Selected Unimportant Words (from {n_unimportant} total):")
print("-" * 40)
for i, (word, importance) in enumerate(zip(unimportant_words, unimportant_importances), 1):
    print(f"{i:2d}. {word:20s} (importance: {importance:.6f})")

Table 2: Feature Importances

Top 10 Most Important Words:
----------------------------------------
 1. great                (importance: 0.030387)
 2. bad                  (importance: 0.021973)
 3. return               (importance: 0.021186)
 4. perfect              (importance: 0.020596)
 5. excel                (importance: 0.019362)
 6. the_best             (importance: 0.018356)
 7. horribl              (importance: 0.018157)
 8. highly_recommend     (importance: 0.017027)
 9. poor                 (importance: 0.017001)
10. your_money           (importance: 0.016887)

10 Randomly Selected Unimportant Words (from 7303 total):
----------------------------------------
 1. alright              (importance: 0.000000)
 2. my_opinion           (importance: 0.000000)
 3. <num>_hour           (importance: 0.000000)
 4. with_<num>           (importance: 0.000000)
 5. it_may               (importance: 0.000000)
 6. same_as              (importance: 0.000000)
 7. of_my                (import

## 2C: Best Random Forest via grid search



This block might take 2-10 minutes. 

If yours runs significantly longer, try this out on Google Colab instead.

In [64]:
# Define hyperparameter grid for random forest
param_grid_forest = {
    'max_features': [3, 10, 33, 100, 333],
    'max_depth': [16, 32],
    'min_samples_leaf': [1],
    'n_estimators': [100],
    'random_state': [101]
}

print("Hyperparameter grid for Random Forest:")
for key, values in param_grid_forest.items():
    print(f"  {key}: {values}")
    
print(f"\nTotal number of combinations: {np.prod([len(v) for v in param_grid_forest.values()])}")

Hyperparameter grid for Random Forest:
  max_features: [3, 10, 33, 100, 333]
  max_depth: [16, 32]
  min_samples_leaf: [1]
  n_estimators: [100]
  random_state: [101]

Total number of combinations: 10


In [65]:
# Create base random forest for grid search
base_forest = sklearn.ensemble.RandomForestClassifier(
    criterion='gini',
    min_samples_split=2,
)

In [66]:
# Create grid search object for random forest
forest_searcher = sklearn.model_selection.GridSearchCV(
    base_forest,
    param_grid_forest,
    scoring='balanced_accuracy',
    cv=my_splitter,
    return_train_score=True,
    refit=False,
    verbose=1,
    n_jobs=-1  # Use all available cores to speed up
)

print("Grid search object created. Ready to fit...")

Grid search object created. Ready to fit...


### Do the search!

In [67]:
# Perform grid search for random forest
print("Starting grid search for Random Forest...")
print("This may take a few minutes...")
start_time = time.time()

forest_searcher.fit(x_tr_va_NF, y_tr_va_N)

elapsed_time = time.time() - start_time
print(f"\nGrid search completed in {elapsed_time:.1f} seconds ({elapsed_time/60:.1f} minutes)!")

Starting grid search for Random Forest...
This may take a few minutes...
Fitting 1 folds for each of 10 candidates, totalling 10 fits

Grid search completed in 71.9 seconds (1.2 minutes)!

Grid search completed in 71.9 seconds (1.2 minutes)!


### Display search results

In [68]:
# Display grid search results
print("Best parameters found:")
print(forest_searcher.best_params_)
print(f"\nBest validation balanced accuracy: {forest_searcher.best_score_:.4f}")

# Create a results dataframe
results_df = pd.DataFrame(forest_searcher.cv_results_)
results_df = results_df.sort_values('rank_test_score')

# Display top 5 configurations
print("\nTop 5 configurations:")
print("-" * 80)
cols_to_show = ['param_max_features', 'param_max_depth', 'mean_train_score', 'mean_test_score', 'rank_test_score']
print(results_df[cols_to_show].head(10).to_string(index=False))

Best parameters found:
{'max_depth': 32, 'max_features': 10, 'min_samples_leaf': 1, 'n_estimators': 100, 'random_state': 101}

Best validation balanced accuracy: 0.8508

Top 5 configurations:
--------------------------------------------------------------------------------
 param_max_features  param_max_depth  mean_train_score  mean_test_score  rank_test_score
                 10               32          0.965501         0.850796                1
                 33               32          0.969600         0.849763                2
                100               16          0.919501         0.847338                3
                100               32          0.968810         0.837029                4
                 33               16          0.927065         0.836302                5
                 10               16          0.920913         0.830739                6
                333               32          0.958386         0.807084                7
               

### Analysis: Random Forest Hyperparameters

**Q1: What is the value of max_features of your best forest?**

The answer will be shown above after the grid search completes.

**Q2: What is the maximum possible value for max_features for this dataset?**

The maximum possible value for max_features is **7729** (the total number of features in the vocabulary). This is because we have 7729 words in our bag-of-words representation.

**Q3: Why is it beneficial to tune this hyperparameter?**

Tuning max_features is beneficial because:
1. **Diversity**: Lower values force each tree to consider only a random subset of features, increasing diversity among trees
2. **Decorrelation**: When trees are built with different feature subsets, they make different errors, and averaging reduces overall variance
3. **Computational Efficiency**: Smaller max_features means faster training per tree
4. **Overfitting Prevention**: Restricting features prevents trees from always using the same strong predictors

**Q4: When fitting random forests, what is the primary tradeoff controlled by the n_estimators hyperparameter?**

The primary tradeoff is between:
- **Performance vs Computational Cost**: More trees generally improve performance (up to a point) but increase training and prediction time
- **Variance Reduction vs Diminishing Returns**: Each additional tree reduces variance, but the benefit decreases as n_estimators grows

**Q5: Can you overfit by setting n_estimators too large? Why or why not?**

**No, you typically cannot overfit** by increasing n_estimators because:
1. Random forests average predictions from many decorrelated trees
2. As n_estimators increases, the model converges to a stable prediction
3. Each tree is trained on a bootstrap sample, providing natural regularization
4. The worst that happens is wasted computation with no improvement, not overfitting
5. However, individual trees can overfit if max_depth is too large or min_samples_leaf is too small

### Build the best random forest using the best hyperparameters found in 2B 

This is necessary so you have the specific best performing forest in your workspace.

Train *only* on training set (do not merge train and valid)


In [69]:
# Build best random forest using the best parameters found
best_forest = base_forest.set_params(**forest_searcher.best_params_)
best_forest.fit(x_tr_NF, y_tr_N)
print("Best random forest trained successfully!")
print(f"\nBest parameters:")
for key, value in forest_searcher.best_params_.items():
    print(f"  {key}: {value}")

Best random forest trained successfully!

Best parameters:
  max_depth: 32
  max_features: 10
  min_samples_leaf: 1
  n_estimators: 100
  random_state: 101


### Table 3: Comparison of methods on the bag-of-words to sentiment classification task.

Please report **balanced accuracy** on the train, valid, and test sets, to 3 digits of precision

**Sample Output** (Feel free to print all values and organize them by hand)

|**method**|**max depth**|**num trees**|**train BAcc**|**valid BAcc**|**test BAcc**|
|:-|:-:|:-:|:-:|:-:|:-:|
|simple Tree	| 1 | 1 | 0.123	|0.456	|0.890|
|best Tree	|1 | 1 | 0.123	|0.456	|0.890|
|simple RandomForest	|1 | 1 | 0.123	|0.456	|0.890|
|best RandomForest	|1 | 1 | 0.123	|0.456	|0.890|

In [70]:
# Evaluate all models and create comparison table
models = {
    'Simple Tree': simple_tree,
    'Best Tree': best_tree,
    'Simple Forest': simple_forest,
    'Best Forest': best_forest
}

# Store results
results = []

for name, model in models.items():
    # Get model parameters
    if 'Tree' in name:
        max_depth = model.max_depth
        n_trees = 1
    else:
        max_depth = model.max_depth
        n_trees = model.n_estimators
    
    # Compute balanced accuracy on all sets
    train_bacc = sklearn.metrics.balanced_accuracy_score(y_tr_N, model.predict(x_tr_NF))
    valid_bacc = sklearn.metrics.balanced_accuracy_score(y_va_N, model.predict(x_va_NF))
    test_bacc = sklearn.metrics.balanced_accuracy_score(y_te_N, model.predict(x_te_NF))
    
    results.append({
        'Method': name,
        'Max Depth': max_depth,
        'Num Trees': n_trees,
        'Train BAcc': f'{train_bacc:.3f}',
        'Valid BAcc': f'{valid_bacc:.3f}',
        'Test BAcc': f'{test_bacc:.3f}'
    })

# Create and display the table
results_table = pd.DataFrame(results)
print("\n" + "=" * 90)
print("Table 3: Comparison of Methods on Bag-of-Words Sentiment Classification")
print("=" * 90)
print(results_table.to_string(index=False))
print("=" * 90)

# Additional analysis
print("\nKey Observations:")
print("-" * 90)
for i, row in enumerate(results):
    if i > 0:
        print(f"\n{row['Method']}:")
        print(f"  - Training: {row['Train BAcc']}, Validation: {row['Valid BAcc']}, Test: {row['Test BAcc']}")
        
# Check for overfitting
print("\n\nOverfitting Analysis:")
print("-" * 90)
for name, model in models.items():
    train_bacc = sklearn.metrics.balanced_accuracy_score(y_tr_N, model.predict(x_tr_NF))
    valid_bacc = sklearn.metrics.balanced_accuracy_score(y_va_N, model.predict(x_va_NF))
    gap = train_bacc - valid_bacc
    print(f"{name:20s}: Train-Valid gap = {gap:.3f}", end="")
    if gap > 0.05:
        print(" (possible overfitting)")
    else:
        print(" (good generalization)")


Table 3: Comparison of Methods on Bag-of-Words Sentiment Classification
       Method  Max Depth  Num Trees Train BAcc Valid BAcc Test BAcc
  Simple Tree          3          1      0.646      0.645     0.646
    Best Tree         32          1      0.909      0.745     0.739
Simple Forest          3        100      0.817      0.812     0.785
  Best Forest         32        100      0.966      0.851     0.853

Key Observations:
------------------------------------------------------------------------------------------

Best Tree:
  - Training: 0.909, Validation: 0.745, Test: 0.739

Simple Forest:
  - Training: 0.817, Validation: 0.812, Test: 0.785

Best Forest:
  - Training: 0.966, Validation: 0.851, Test: 0.853


Overfitting Analysis:
------------------------------------------------------------------------------------------
Simple Tree         : Train-Valid gap = 0.001 (good generalization)
Best Tree           : Train-Valid gap = 0.164 (possible overfitting)
Simple Forest       : Train

### Conclusions

Based on the results in Table 3, we can draw the following conclusions:

1. **Random Forests Outperform Single Trees**: 
   - Both simple and best random forests achieve higher accuracy than their decision tree counterparts
   - This demonstrates the power of ensemble methods

2. **Hyperparameter Tuning Helps**:
   - The best tree performs better than the simple tree
   - The best forest performs better than the simple forest
   - This shows the importance of proper hyperparameter selection

3. **Generalization Performance**:
   - By comparing train vs. validation/test scores, we can assess overfitting
   - Random forests typically show better generalization due to ensemble averaging
   - Decision trees with high depth may overfit the training data

4. **Test Set Performance**:
   - The test set balanced accuracy gives us the best estimate of real-world performance
   - The best random forest should achieve the highest test accuracy
   - This confirms that random forests are well-suited for text classification tasks

5. **Practical Implications**:
   - For production use, the best random forest is the recommended model
   - The tradeoff is computational cost (100 trees vs 1 tree)
   - The performance gain usually justifies the additional computational expense